In [155]:
import pandas as pd
import re
import evaluate
import nltk
from nltk.tokenize import sent_tokenize
import numpy as np
from transformers import AutoTokenizer

In [132]:
with open("app2.log",'r') as f:
    lines = f.readlines()
    fixed_logs = list()
    for line in lines:
        if line.startswith('<br/>') and fixed_logs:
            fixed_logs[-1] = fixed_logs[-1].strip() + line
        else:
            fixed_logs.append(line)

    data = []
    pattern = r'^(\S+) INFO \[user\]:\[(.*?)\]\|(.*)$'

    for log in fixed_logs:
        match = re.match(pattern, log)
        if match:
            timestamp = match.group(1)
            model = match.group(2)
            fields = match.group(3).split('|')

            data.append([timestamp, model] + fields)

    columns = [
        'timestamp', 'model', 'review', 'numerical_rate',
        'translated_text', 'alphabet', 'lang', 'summary', 'metric'
    ]

    df = pd.DataFrame(data, columns=columns)

    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df['numerical_rate'] = pd.to_numeric(df['numerical_rate'], errors='coerce')
    df.drop('metric', axis=1)

In [133]:
df = df[df.alphabet == "LATIN"]
df

,timestamp,model,review,numerical_rate,translated_text,alphabet,lang,summary,metric
0,2026-01-15 20:20:29.838590+00:00,base,"We spent a lovely time in Charles' BnB, always...",5,"We spent a lovely time in Charles' BnB, always...",LATIN,en,"Charles is a great host, very friendly and car...",0.0
1,2026-01-15 20:20:30.246873+00:00,base,very good location and excellent transportatio...,5,very good location and excellent transportatio...,LATIN,en,very good location and excellent transportatio...,0.0
2,2026-01-15 20:20:30.664087+00:00,base,Dan was super helpful and understanding. His a...,5,Dan was super helpful and understanding. His a...,LATIN,en,Dan was super helpful and understanding. His a...,0.0
3,2026-01-15 20:20:31.209211+00:00,complex,"The landlords are very friendly and welcoming,...",5,"The landlords are very friendly and welcoming,...",LATIN,en,"The landlords are very friendly and welcoming,...",0.0
4,2026-01-15 20:20:31.809739+00:00,base,The flat is located in a really trendy neighbo...,5,The flat is located in a really trendy neighbo...,LATIN,en,The flat is located in a really trendy neighbo...,0.0
...,...,...,...,...,...,...,...,...,...
1725,2026-01-16 19:47:10.667275+00:00,complex,Appartement très bien placé: 100 m d’une stati...,3,Very well located apartment: 100 m from a Metr...,LATIN,fr,The apartment is well-located but has some noi...,0.0
1726,2026-01-16 19:47:11.652353+00:00,complex,El apartamento de Omar se ajustaba a lo descri...,2,Omar's apartment fit the description. We staye...,LATIN,es,"Omar's apartment was well-equipped, comfortabl...",0.0
1727,2026-01-16 19:47:12.434183+00:00,complex,Excellent séjour à Londres . Appartement très ...,4,Great stay in London. Very well decorated apar...,LATIN,fr,Guest had a great stay in London with a well-d...,0.0
1728,2026-01-16 19:47:12.899996+00:00,base,This has been my second visit to Nadia's apart...,5,This has been my second visit to Nadia's apart...,LATIN,en,This has been my second visit to Nadia's apart...,0.0


In [134]:
print(df.lang.value_counts())
print(df.model.value_counts())
print(df.numerical_rate.value_counts())

lang
en    1519
fr      90
de      60
es      40
it      18
Name: count, dtype: int64
model
base       869
complex    858
Name: count, dtype: int64
numerical_rate
5    1405
4     102
0      82
2      76
3      34
1      28
Name: count, dtype: int64


In [135]:
df_base = df[df['model'] == 'base']
df_complex = df[df['model'] == 'complex']

# Przykładowe streszczenia

In [145]:
print(df_base.iloc[0].translated_text)
print()
print(df_base.iloc[0].summary)

We spent a lovely time in Charles' BnB, always welcoming and refreshing after long days visiting London. The room completely met our needs and is truly delicious, well-finished and furnished. Charles (and his wife too) is a great host, very friendly and caring about your stay at his place and your experience in London. His wife gave us a lot of very helpful suggestions. They were both very prompt in meeting our needs and we really appreciated the fact that they allowed us to store our luggage in their home well beyond the check-out time, to help us enjoying our last day. Highly recommended!

Charles is a great host, very friendly and caring about your stay at his place. His wife gave us a lot of very helpful suggestions. They were both very prompt in meeting our needs.


In [146]:
print(df_complex.iloc[0].translated_text)
print()
print(df_complex.iloc[0].summary)

The landlords are very friendly and welcoming, making me feel at home. Everything is clean and tidy. You can cook there, and they set the most basic stuff for you. The surroundings and transportation are also very nice and convenient. In front of the house, there’s a park you can walk. Besides, It's only several mins walk to DLR, bus, train station and near the tube station. You can go to the centre of London in half an hour. There are plenty of shopping markets nearby, and you can buy daily necessities and food there if you need them.

The landlords are very friendly and welcoming, making the guest feel at home with their clean and convenient location.


In [152]:
rouge = evaluate.load("rouge")
nltk.download('punkt_tab')
nltk.download('punkt')

def compute_rouge(predictions, references, tokenizer):
    decoded_preds = ["\n".join(sent_tokenize(pred.strip())) for pred in predictions]
    decoded_labels = ["\n".join(sent_tokenize(label.strip())) for label in references]

    result = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True,
    )

    result = {k: round(v * 100, 4) for k, v in result.items()}

    inputs = tokenizer(predictions, padding=True, truncation=True, return_tensors="np")
    prediction_tokens = inputs["input_ids"]

    prediction_lens = np.sum(prediction_tokens != tokenizer.pad_token_id, axis=1)
    result["gen_len"] = np.mean(prediction_lens)

    return result

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/vistek528/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt to /home/vistek528/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [161]:
base_tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large-cnn")
complex_tokenizer = AutoTokenizer.from_pretrained("./ium_bart_final_v3")
base_results = compute_rouge(
    df_base['summary'].tolist(),
    df_base['translated_text'].tolist(),
    base_tokenizer
)

complex_results = compute_rouge(
    df_complex['summary'].tolist(),
    df_complex['translated_text'].tolist(),
    complex_tokenizer
)

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


In [162]:
base_results

{'rouge1': np.float64(60.3948),
 'rouge2': np.float64(58.1367),
 'rougeL': np.float64(59.6875),
 'rougeLsum': np.float64(60.4034),
 'gen_len': np.float64(45.388952819332566)}

In [163]:
complex_results

{'rouge1': np.float64(29.9079),
 'rouge2': np.float64(14.6601),
 'rougeL': np.float64(23.9482),
 'rougeLsum': np.float64(26.652),
 'gen_len': np.float64(26.206293706293707)}

Jak można zauważyć na porównaniu wyników testów AB, model bazowy posiadał wyższe wartości miary jakości, liczone tym razem w porównaniu do oryginalnej recenzji (lub jej tłumaczenia). Nie jest to jednak jednoznaczne, z wyższą jakością takich streszczeń, ponieważ miara ROUGE nie bierze pod uwagę m.in. parafrazowania, przez co wyniki mogą być zaniżone. Ponadto model wytrenowany na potrzeby przedmiotu był uczony generowania bardzo krótkich, ale treściwych streszczeń, które mają przekazać esencję oryginalnej opinii użytkownika. Przez to, że są krótsze (wg wskaźnika gen_len średnio są o prawie 20 tokenów krótsze) to do streszczenia przenika mniej słów z pierwotnej recenzji.

Podsumowując, model zaawansowany spełnił dobrze postawione przed nim zadanie w postaci istotnego streszczania oryginalnych recenzji celem zaoszczędzenia czasu właścicieli poszczególnych lokali w serwisie nocarz. Wychwytuje on najistotniejsze fakty z recenzji użytkownika, takie jak wzmianka o dobrej lokalizacji, miłej obsłudze czy niedociągnięcia w wyposażeniu lokalu przekazując skondensowaną wiedzę właścicielowi, który dzięki temu może poprawić jakość świadczonych usług.